# 02 — Global Discrepancy (RQ1)

**Question.** At the global level (2014–2025), do the annual trends of ACLED *events* and *fatalities* diverge?

**Method.** Aggregate cleaned weekly data to annual global totals; visualise with (a) a cumulative world map of events and fatalities, (b) a dual-axis line chart, and (c) a lethality ratio. Quantify divergence with Pearson/Spearman correlations, year-over-year growth rates, and an OLS comparison of log-linear trend slopes with an interaction test for whether the slopes differ.

**Takeaway.** Reported at the bottom of the notebook.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

PROJECT = Path('/Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5')
DERIVED = PROJECT / 'data' / 'derived'
FIGDIR = PROJECT / 'docs' / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DERIVED / 'acled_clean.parquet')
print('Loaded df_weekly:', df.shape)

Loaded df_weekly: (838798, 15)


## Annual global aggregates

In [2]:
annual = (df.groupby('YEAR')[['EVENTS','FATALITIES']].sum()
            .astype(int)
            .reset_index())
annual['LETHALITY'] = annual['FATALITIES'] / annual['EVENTS']
annual['EVENTS_YOY'] = annual['EVENTS'].pct_change()
annual['FATALITIES_YOY'] = annual['FATALITIES'].pct_change()
annual

,YEAR,EVENTS,FATALITIES,LETHALITY,EVENTS_YOY,FATALITIES_YOY
0,2014,24662,47069,1.908564,NaN,NaN
1,2015,36222,62546,1.726741,0.468737,0.328815
2,2016,76458,124168,1.624003,1.110817,0.985227
3,2017,114165,184837,1.619034,0.493173,0.488604
4,2018,213135,178908,0.839412,0.866903,-0.032077
5,2019,231625,153678,0.663478,0.086753,-0.141022
6,2020,268297,139323,0.519286,0.158325,-0.093410
7,2021,293601,160264,0.545856,0.094313,0.150305
8,2022,326622,172326,0.527601,0.112469,0.075263
9,2023,353771,200863,0.567777,0.083121,0.165599


## World map: cumulative events and fatalities by country (2014–2025)

Two side-by-side choropleths. Country names in ACLED are passed directly to Plotly using `locationmode='country names'`.

In [3]:
by_country = (df.groupby('COUNTRY')[['EVENTS','FATALITIES']].sum().reset_index())

fig_map = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'choropleth'}, {'type':'choropleth'}]],
    subplot_titles=('Cumulative EVENTS, 2014–2025', 'Cumulative FATALITIES, 2014–2025')
)
fig_map.add_trace(go.Choropleth(
    locations=by_country['COUNTRY'], locationmode='country names',
    z=np.log10(by_country['EVENTS'].clip(lower=1)),
    colorscale='Blues', colorbar=dict(title='log10(events)', x=0.46),
    customdata=np.stack([by_country['COUNTRY'], by_country['EVENTS']], axis=-1),
    hovertemplate='%{customdata[0]}<br>Events: %{customdata[1]:,}<extra></extra>',
    name='Events'
), row=1, col=1)
fig_map.add_trace(go.Choropleth(
    locations=by_country['COUNTRY'], locationmode='country names',
    z=np.log10(by_country['FATALITIES'].clip(lower=1)),
    colorscale='Reds', colorbar=dict(title='log10(fatalities)', x=1.02),
    customdata=np.stack([by_country['COUNTRY'], by_country['FATALITIES']], axis=-1),
    hovertemplate='%{customdata[0]}<br>Fatalities: %{customdata[1]:,}<extra></extra>',
    name='Fatalities'
), row=1, col=2)
fig_map.update_layout(
    title='Cumulative ACLED activity by country, 2014–2025 (log10 colour scale)',
    height=520, margin=dict(l=10,r=10,t=80,b=10)
)
fig_map.write_html(FIGDIR / '02_world_map.html', include_plotlyjs='cdn')
fig_map.show()

## Dual-axis line chart: annual events vs. fatalities

In [4]:
fig_dual = make_subplots(specs=[[{'secondary_y': True}]])
fig_dual.add_trace(go.Scatter(x=annual['YEAR'], y=annual['EVENTS'], name='Events',
                              mode='lines+markers', line=dict(color='steelblue', width=3)),
                   secondary_y=False)
fig_dual.add_trace(go.Scatter(x=annual['YEAR'], y=annual['FATALITIES'], name='Fatalities',
                              mode='lines+markers', line=dict(color='firebrick', width=3)),
                   secondary_y=True)
fig_dual.update_yaxes(title_text='Events (count)', secondary_y=False)
fig_dual.update_yaxes(title_text='Fatalities (count)', secondary_y=True)
fig_dual.update_xaxes(title_text='Year', dtick=1)
fig_dual.update_layout(title='Global annual ACLED events vs. fatalities, 2014–2025',
                       height=450, hovermode='x unified')
fig_dual.write_html(FIGDIR / '02_dual_axis.html', include_plotlyjs='cdn')
fig_dual.show()

## Lethality ratio (fatalities per event)

In [5]:
fig_leth = go.Figure()
fig_leth.add_trace(go.Scatter(x=annual['YEAR'], y=annual['LETHALITY'],
                              mode='lines+markers', line=dict(color='purple', width=3),
                              name='Fatalities / event'))
fig_leth.update_layout(title='Global lethality ratio: fatalities per event, 2014–2025',
                       xaxis_title='Year', yaxis_title='Fatalities per event',
                       height=420, xaxis=dict(dtick=1))
fig_leth.write_html(FIGDIR / '02_lethality.html', include_plotlyjs='cdn')
fig_leth.show()

## Quantifying divergence

1. Pearson & Spearman correlation between annual events and fatalities.
2. Year-over-year growth rates (already computed above).
3. OLS of `log(Y)` on `YEAR` for both series, then a pooled regression with a `series × year` interaction to test whether the two slopes differ statistically.

In [6]:
pear = stats.pearsonr(annual['EVENTS'], annual['FATALITIES'])
spear = stats.spearmanr(annual['EVENTS'], annual['FATALITIES'])
print(f'Pearson  r = {pear.statistic:.3f}, p = {pear.pvalue:.4g}')
print(f'Spearman r = {spear.statistic:.3f}, p = {spear.pvalue:.4g}')
print('\nYoY growth (mean, median):')
print('  Events     mean={:.1%}  median={:.1%}'.format(
    annual['EVENTS_YOY'].mean(), annual['EVENTS_YOY'].median()))
print('  Fatalities mean={:.1%}  median={:.1%}'.format(
    annual['FATALITIES_YOY'].mean(), annual['FATALITIES_YOY'].median()))

Pearson  r = 0.856, p = 0.0003826
Spearman r = 0.811, p = 0.001363

YoY growth (mean, median):
  Events     mean=33.0%  median=11.2%
  Fatalities mean=19.5%  median=15.0%


In [7]:
def ols_log_slope(y, x):
    res = stats.linregress(x, np.log(y))
    return {'slope': res.slope, 'intercept': res.intercept,
            'r2': res.rvalue**2, 'p': res.pvalue, 'se': res.stderr}

x = annual['YEAR'].to_numpy()
ev = ols_log_slope(annual['EVENTS'].to_numpy(), x)
ft = ols_log_slope(annual['FATALITIES'].to_numpy(), x)
print('log(EVENTS) ~ YEAR')
print(f"  slope={ev['slope']:.4f}  SE={ev['se']:.4f}  R2={ev['r2']:.3f}  p={ev['p']:.3g}")
print(f"  annualised growth = {(np.exp(ev['slope'])-1)*100:.1f}% / yr")
print('log(FATALITIES) ~ YEAR')
print(f"  slope={ft['slope']:.4f}  SE={ft['se']:.4f}  R2={ft['r2']:.3f}  p={ft['p']:.3g}")
print(f"  annualised growth = {(np.exp(ft['slope'])-1)*100:.1f}% / yr")

log(EVENTS) ~ YEAR
  slope=0.2428  SE=0.0332  R2=0.843  p=2.55e-05
  annualised growth = 27.5% / yr
log(FATALITIES) ~ YEAR
  slope=0.1158  SE=0.0251  R2=0.681  p=0.00095
  annualised growth = 12.3% / yr


In [8]:
# Pooled OLS with series × year interaction.
# Model: log(Y) = b0 + b1*YEAR + b2*IS_FAT + b3*(IS_FAT*YEAR) + eps
# The interaction coefficient b3 tests whether the fatalities slope differs from the events slope.
long = pd.concat([
    pd.DataFrame({'YEAR': x, 'logY': np.log(annual['EVENTS']),     'IS_FAT': 0}),
    pd.DataFrame({'YEAR': x, 'logY': np.log(annual['FATALITIES']), 'IS_FAT': 1}),
], ignore_index=True)
X = np.column_stack([np.ones(len(long)), long['YEAR'], long['IS_FAT'], long['YEAR']*long['IS_FAT']])
y = long['logY'].to_numpy()
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta
n, k = X.shape
sigma2 = (resid @ resid) / (n - k)
cov = sigma2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov))
names = ['Intercept', 'YEAR', 'IS_FAT', 'YEAR:IS_FAT']
print('Pooled regression log(Y) ~ YEAR * IS_FAT')
print(f'{"term":<14} {"coef":>10} {"SE":>9} {"t":>7}   p')
for name, b, s in zip(names, beta, se):
    t = b / s
    p = 2 * (1 - stats.t.cdf(abs(t), df=n-k))
    print(f'{name:<14} {b:>10.4f} {s:>9.4f} {t:>7.2f}   {p:.4g}')
print('\n→ The YEAR:IS_FAT coefficient is the DIFFERENCE in log-slope (fatalities minus events).')

Pooled regression log(Y) ~ YEAR * IS_FAT
term                 coef        SE       t   p
Intercept       -478.3017   59.3793   -8.06   1.048e-07
YEAR               0.2428    0.0294    8.26   7.116e-08
IS_FAT           256.3981   83.9750    3.05   0.006275
YEAR:IS_FAT       -0.1270    0.0416   -3.06   0.006249

→ The YEAR:IS_FAT coefficient is the DIFFERENCE in log-slope (fatalities minus events).


## Takeaway

A narrative takeaway will be drafted in the report after Step 3. Numerical findings from this notebook (annualised growth rates, slope difference, lethality trend) are the inputs to the RQ1 section of the report.